In [0]:
create or refresh streaming table circuitbox.lakehouse.bronze_customers
tblproperties('quality' = 'bronze')
as 
select *,
       _metadata.file_path as input_file_path,
       current_timestamp() as ingestion_timestamp
from stream cloud_files(
    '/Volumes/circuitbox/landing/operational_data/customers/',
    'json',
    map('cloudFiles.inferColumnTypes', 'true')
);

In [0]:
create or refresh streaming table circuitbox.lakehouse.silver_customers_clean(
    constraint valid_customer_id expect (customer_id is not null) on violation fail update,
    constraint valid_customer_name expect(customer_name is not null) on violation drop row,
    constraint valid_telephone expect(length(telephone) >=10),
    constraint valid_email expect(email is not null),
    constraint valid_date_of_birth expect(date_of_birth >= '1920-01-01')
)
tblproperties('quality' = 'silver')
as
select customer_id,
       customer_name,
       date_of_birth::date as date_of_birth,
       telephone,
       email,
       created_date::date as created_date
from stream (circuitbox.lakehouse.bronze_customers);

In [0]:
create or refresh streaming table circuitbox.lakehouse.silver_customers
tblproperties('quality' = 'silver');

In [0]:
apply changes into circuitbox.lakehouse.silver_customers
from stream (circuitbox.lakehouse.silver_customers_clean)
keys (customer_id)
sequence by created_date
stored as scd type 1;